# Spam Detection - Machine Learning Assignment

## Student Details
- **Name:** Ali Valiyev.
- **ID:** 5584048

## Dataset
- **Name:** Spam Text Message Classification
- **Source:** https://www.kaggle.com/datasets/team-ai/spam-text-message-classification
- **Task:** Binary Classification — detect whether an SMS message is spam or not (ham)

## AI Assistance Used
- Used ChatGPT / Claude to understand feature engineering concepts (TF-IDF, tokenization)

In [5]:
!pip install scikit-learn

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [15]:
#dataset
df = pd.read_csv("spam.csv", encoding="latin-1")

df.columns = ["label", "text"]

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
# spam=1, ham=0
df["label"] = df["label"].map({"spam": 1, "ham": 0})

# (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

Training samples: 4457
Test samples:     1115


## Part 2 - Feature Engineering

Converting raw text into numbers using **TF-IDF (Term Frequency - Inverse Document Frequency)**.
- TF: how often a word appears in a message
- IDF: how rare the word is across all messages
- Words like "free", "win", "prize" will score high in spam messages

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF converter
tfidf = TfidfVectorizer(max_features=3000)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print(f"Each message is now represented by {X_train_tfidf.shape[1]} numbers")
print(f"Training set shape: {X_train_tfidf.shape}")
print(f"Test set shape:     {X_test_tfidf.shape}")

Each message is now represented by 3000 numbers
Training set shape: (4457, 3000)
Test set shape:     (1115, 3000)


In [19]:
# first 3 messages after feature engineering
sample_messages = X_train.head(3)
sample_tfidf = tfidf.transform(sample_messages)

for i, (message, vector) in enumerate(zip(sample_messages, sample_tfidf)):
    # the top 5 words with highest TF-IDF scores in this message
    feature_names = tfidf.get_feature_names_out()
    scores = vector.toarray()[0]
    top_indices = scores.argsort()[-5:][::-1]
    top_words = [(feature_names[j], round(scores[j], 3)) for j in top_indices]
    
    print(f"--- Message {i+1} ---")
    print(f"Text: {message[:80]}...")
    print(f"Top 5 words (word, score): {top_words}")
    print()

--- Message 1 ---
Text: Reply to win Â£100 weekly! Where will the 2006 FIFA World Cup be held? Send STOP...
Top 5 words (word, score): [('87239', np.float64(0.354)), ('cup', np.float64(0.339)), ('to', np.float64(0.297)), ('weekly', np.float64(0.285)), ('end', np.float64(0.273))]

--- Message 2 ---
Text: Hello. Sort of out in town already. That . So dont rush home, I am eating nachos...
Top 5 words (word, score): [('eta', np.float64(0.365)), ('rush', np.float64(0.343)), ('eating', np.float64(0.322)), ('sort', np.float64(0.308)), ('town', np.float64(0.276))]

--- Message 3 ---
Text: How come guoyang go n tell her? Then u told her?...
Top 5 words (word, score): [('her', np.float64(0.669)), ('told', np.float64(0.379)), ('tell', np.float64(0.317)), ('then', np.float64(0.287)), ('come', np.float64(0.287))]

